In [54]:
import sys
sys.path.append(r"C:\\Program Files\\Lumerical\\v232\\api\\python")
import lumapi

device = lumapi.DEVICE("ln_eom.ldev")

In [55]:
import importlib
import eom_render as eomr
import charge
import feem
importlib.reload(eomr) # force reload since notebook doesn't detect changes in py file
importlib.reload(charge)
importlib.reload(feem)

device.switchtolayout()
device.deleteall()
eomr.add_materials(device)
eomr.draw_eom(device)
charge.add_charge_solver(device)
feem.add_feem_solver(device)
# device.save("ln_eom.ldev")run("FEEM")

In [ ]:
# Run CHARGE simulations and find index perturbation
import numpy as np
import matplotlib.pyplot as plt

device.switchtolayout()
device.run("CHARGE")

def get_E_field(device):
    # result_names = device.getresult("CHARGE::monitor")
    # print(result_names)
    electrostatics_res = device.getresult("CHARGE::monitor", "electrostatics")
    E = np.squeeze(electrostatics_res['E'])
    x = np.squeeze(electrostatics_res['x'])
    z = np.squeeze(electrostatics_res['z'])
    elements = device.getdata("CHARGE::monitor", "electrostatics", "elements")
    return E, x, z, elements

def get_index_perturbation(E):
    # LN telecom permittivity
    eps_o = 2.21**2
    eps_e = 2.14**2 
    # LN pockels tensor
    r_13 = 9.6e-12
    r_33 = 30.9e-12

    # Pockels effect (see lumerical example for math)
    deps_inv = {
        'x': r_33 * E[:, 0],
        'y': r_13 * E[:, 0],
        'z': r_13 * E[:, 0]
    }
    
    # new total refractive index (original + perturbed)
    n_EO_x = ((1 / eps_e) + deps_inv['x']) ** (-0.5)
    n_EO_y = ((1 / eps_e) + deps_inv['y']) ** (-0.5)
    n_EO_z = ((1 / eps_e) + deps_inv['z']) ** (-0.5)

    # change in refractive index
    dn = {
        'x': n_EO_x - (eps_e ** 0.5),
        'y': n_EO_y - (eps_o ** 0.5),
        'z': n_EO_z - (eps_o ** 0.5),
    }
    n_EO = [n_EO_x, n_EO_y, n_EO_z] 
    return n_EO, dn

E, x, z, elements = get_E_field(device)
n_EO, dn = get_index_perturbation(E)

<class 'dict'>
[4.69818525e-05 5.26858690e-05 6.00413163e-05 ... 6.30685983e-08
 2.57488932e-08 6.93938307e-08]
[-0.0699854  -0.06998363 -0.06998135 ... -0.06999998 -0.06999999
 -0.06999998]
[-0.0699854  -0.06998363 -0.06998135 ... -0.06999998 -0.06999999
 -0.06999998]


In [ ]:
from scipy.io import savemat

# Build dn array in shape (N, 3)
dn_tri = np.zeros((len(x), 3))
dn_tri[:, 0] = dn['x']
dn_tri[:, 1] = dn['y']
dn_tri[:, 2] = dn['z']

# Build y (if it's a 2D simulation, just use zeros)
y = np.zeros_like(x)

# Create dictionary to save
data = {
    'dn': dn_tri,
    'x': x,
    'y': y,
    'z': z,
    'elements': elements
}

# Save to .mat file
save_path = "nk_region.mat"  # adjust as needed
savemat(save_path, data)
print(f"✅ Saved data to {save_path}")


Saved nk_region.mat successfully.


In [67]:
import numpy as np
from scipy.io import savemat

def run_ln_phase_modulator_single_voltage(device, signal_voltage=5.0, save_path="nk_region.mat"):
    """
    Run CHARGE simulation and compute index perturbation due to Pockels effect
    at a single voltage (default = 5V). Save result to .mat file for FEEM use.

    Parameters:
        device: Lumerical DEVICE object (via lumapi)
        signal_voltage: Voltage at which EO modulation is computed
        save_path: Output .mat file for unstructureddataset

    Returns:
        dn: delta-n (index perturbation), shape (N, 3)
        n_EO: total index, shape (N, 3)
    """
    # --- Constants: LN material parameters ---
    eps_o = 2.21 ** 2
    eps_e = 2.14 ** 2
    r_13 = 9.6e-12
    r_33 = 30.9e-12

    # --- Run CHARGE simulation ---
    print("🔌 Running CHARGE simulation...")
    device.switchtolayout()
    device.run("CHARGE")

    # --- Extract E-field result ---
    electro = device.getresult("CHARGE::monitor", "electrostatics")
    E = np.squeeze(electro['E'])  # (N, 3)
    x = np.squeeze(electro['x'])
    z = np.squeeze(electro['z'])
    y = np.zeros_like(x)  # if 2D simulation
    elements = electro['elements']  # triangle connectivity for unstructureddataset

    if E.ndim != 2 or E.shape[1] != 3:
        raise ValueError("E-field shape invalid. Expected (N, 3), got: " + str(E.shape))

    N = E.shape[0]
    E_x = E[:, 0]  # Only Ex is used for this formulation

    # --- Apply Pockels effect ---
    deps_inv = np.zeros((N, 3))
    deps_inv[:, 0] = r_33 * E_x  # x
    deps_inv[:, 1] = r_13 * E_x  # y
    deps_inv[:, 2] = r_13 * E_x  # z

    eps_unperturbed = np.zeros((N, 3))
    eps_unperturbed[:, 0] = eps_e
    eps_unperturbed[:, 1] = eps_o
    eps_unperturbed[:, 2] = eps_o

    eps_eff_inv = (1 / eps_unperturbed) + deps_inv
    n_EO = np.sqrt(1 / eps_eff_inv)

    # --- Compute delta-n ---
    dn = np.zeros_like(n_EO)
    dn[:, 0] = n_EO[:, 0] - np.sqrt(eps_e)  # x
    dn[:, 1] = n_EO[:, 1] - np.sqrt(eps_o)  # y
    dn[:, 2] = n_EO[:, 2] - np.sqrt(eps_o)  # z

    # --- Prepare output data ---
    print(f"💾 Saving EO modulation results to {save_path}...")
    data = {
        'dn': dn,            # Δn values per node
        'n_EO': n_EO,        # Total index (optional)
        'x': x,
        'y': y,
        'z': z,
        'elements': elements
    }
    savemat(save_path, data)
    print("✅ EO index change data saved successfully.")

    return dn, n_EO
dn, n_EO = run_ln_phase_modulator_single_voltage(device, signal_voltage=5.0)

🔌 Running CHARGE simulation...


KeyError: 'elements'